# Psoriasis scRNA-seq Drug Repurposing Pipeline
**Transcriptome-Guided Drug Repurposing for Psoriasis**

Glen Ritschel | Ritschel Research | 2026

GitHub: https://github.com/glenritschel/psoriasis-scrna

**Dataset**: GSE173706 — 33 samples across NS (normal skin), PN (uninvolved psoriatic),
PP (lesional psoriatic). CSV format, Ensembl gene IDs.

**Primary comparison**: PP (lesional) vs PN (uninvolved) — paired disease-specific DE

**Before running**: Runtime > Change runtime type > T4 GPU

## Step 1: Clone repo and install dependencies

In [ ]:
import os
REPO_URL = "https://github.com/glenritschel/psoriasis-scrna"
REPO_DIR = "psoriasis-scrna"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull
%cd /content/{REPO_DIR}/notebooks
print("Working directory:", os.getcwd())

In [ ]:
import subprocess, sys
packages = [
    "scvi-tools==1.3.3", "scanpy==1.11.5", "gseapy==1.1.10",
    "leidenalg", "python-igraph", "scikit-misc", "biopython", "pybiomart"
]
for pkg in packages:
    print("Installing", pkg, "...")
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
print("All dependencies installed.")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Download GSE173706 from NCBI GEO

All 33 samples are bundled in a single 252MB tar file at the series level.
Files are in CSV format (genes x cells, Ensembl IDs).

In [ ]:
import os, urllib.request, tarfile, glob

RAW_DIR = "/content/psoriasis-scrna/data/raw/GSE173706"
os.makedirs(RAW_DIR, exist_ok=True)

TAR_URL = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE173nnn/GSE173706/suppl/GSE173706_RAW.tar"
TAR_PATH = os.path.join(RAW_DIR, "GSE173706_RAW.tar")

if not os.path.exists(TAR_PATH):
    print("Downloading GSE173706_RAW.tar (252MB)...", flush=True)
    urllib.request.urlretrieve(TAR_URL, TAR_PATH)
    print("Download complete.", round(os.path.getsize(TAR_PATH) / 1e6, 0), "MB")
else:
    print("Already downloaded.")

if len(glob.glob(os.path.join(RAW_DIR, "*.csv.gz"))) < 10:
    print("Extracting...", flush=True)
    with tarfile.open(TAR_PATH, filter="data") as tf:
        tf.extractall(RAW_DIR)
    print("Extraction complete.")

csv_files = sorted(glob.glob(os.path.join(RAW_DIR, "GSM*.csv.gz")))
print(f"CSV files ready: {len(csv_files)}")
for cond in ["PP", "PN", "NS"]:
    n = sum(1 for f in csv_files if f"_{cond}-" in os.path.basename(f))
    print(f"  {cond}: {n} samples")

## Step 3: Download Ensembl gene ID mapping

GSE173706 uses Ensembl IDs (ENSG...). This cell maps them to HGNC gene symbols
via Ensembl BioMart. Takes ~1 minute. Result is cached to disk.

In [ ]:
import os, pandas as pd
MAPPING_PATH = "/content/psoriasis-scrna/data/raw/GSE173706/ensembl_to_symbol.csv"

if not os.path.exists(MAPPING_PATH):
    from pybiomart import Dataset
    print("Querying Ensembl BioMart...")
    ds = Dataset(name="hsapiens_gene_ensembl", host="http://www.ensembl.org")
    mapping = ds.query(attributes=["ensembl_gene_id", "hgnc_symbol"])
    mapping.to_csv(MAPPING_PATH, index=False)
    print("Saved:", len(mapping), "entries ->", MAPPING_PATH)
else:
    mapping = pd.read_csv(MAPPING_PATH)
    print("Already present:", len(mapping), "entries")
print(mapping.head())

## Step 4: Script 01 — Load, QC, Condition Labelling

In [ ]:
%run ../src/01_load_qc.py

## Step 5: Script 02 — scVI Embedding + Leiden Clustering (GPU)

In [ ]:
%run ../src/02_scvi_embed.py

## Step 6: Script 03 — Cell Type Annotation

In [ ]:
%run ../src/03_annotate_clusters.py

## Step 7: Script 04 — Psoriasis Signature Scoring

In [ ]:
%run ../src/04_signature_scoring.py

## Step 8: Script 05 — Differential Expression (PP vs PN)

In [ ]:
%run ../src/05_differential_expression.py

## Step 9: Script 06 — LINCS L1000 Reversal Scoring

In [ ]:
%run ../src/06_lincs_repurposing.py

## Step 10: Script 07 — Novelty Assessment + Priority Scoring

In [ ]:
%run ../src/07_novelty_prioritization.py

## Step 11: Save outputs to Google Drive

**Run this cell immediately after script 07 completes** — do not wait or the
runtime may time out and the processed files will be lost.

In [ ]:
from google.colab import drive
import shutil, os, glob
drive.mount("/content/drive")
DRIVE_OUTPUT = "/content/drive/MyDrive/Ritschel_Research/psoriasis_scrna_output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
outputs = (
    glob.glob("/content/psoriasis-scrna/data/processed/*.csv") +
    glob.glob("/content/psoriasis-scrna/data/processed/*.h5ad")
)
for f in outputs:
    shutil.copy2(f, os.path.join(DRIVE_OUTPUT, os.path.basename(f)))
    print("Saved:", os.path.basename(f))
print("All outputs saved to", DRIVE_OUTPUT)